# Real-Time LangGraph Multi-Agent Travel Planner — Explained

This notebook builds a **multi-agent AI system** using **LangGraph**. Instead of one LLM doing everything, the work is split across small, specialised "agents" (just Python functions), each responsible for one job:

- **Supervisor agent** — turns the user's plain-English request into structured data (origin, destination, dates, travelers, preferences).
- **Flight agent** — calls a live flights API (Aviationstack).
- **Hotel agent** — calls a live web-search API (Tavily) for hotel options.
- **News agent** — calls a live news API (NewsData.io) to check for disruptions/events at the destination.
- **Weather agent** — calls a live weather API (OpenWeather).
- **Cost agent** — combines gathered data with fresh web research and asks the LLM to produce a cost estimate.
- **Itinerary agent** — asks the LLM to turn all collected data into a day-by-day plan.
- **Final agent** — asks the LLM to assemble everything into one polished answer for the user.

LangGraph wires these functions into a **graph**: a flowchart where each node is an agent and each edge says "after this agent finishes, run that one next." The key concept is the **shared state** — one dictionary-like object every agent reads from and writes back into.

A markdown cell explaining each step has been inserted directly before the corresponding code cell below. Original outputs have been cleared so you can run the notebook fresh with your own API keys.

**Key ideas to watch for as you go:**
1. State is the backbone connecting otherwise-independent agent functions.
2. Prompts are explicitly "grounded" with real retrieved data and told not to invent facts (a RAG pattern).
3. LLM responses need defensive parsing since content can come back as a string or a list of blocks.
4. The graph's edges create parallelism: independent agents run concurrently, dependent ones wait.
5. Every agent has an isolated test cell — a good debugging practice for multi-agent systems.


### Cell: Install dependencies

- `!pip -q install -U ...` runs a shell command (the `!` tells Jupyter/Colab this isn't Python) to install/upgrade the libraries this notebook needs:
  - **`langgraph`** — the graph/orchestration framework used to wire the agents together.
  - **`langchain`** and **`langchain-google-genai`** — LangChain core plus the adapter for calling Google's Gemini models.
  - **`tavily-python`** — client for the Tavily web-search API.
  - **`requests`** — standard HTTP library, used for the other REST APIs called directly.
- `-q` = quiet (less console noise), `-U` = upgrade to the latest version if already installed.

In [ ]:
!pip -q install -U langgraph langchain langchain-google-genai tavily-python requests

### Cell: Load API keys securely

- `os` lets us read/write **environment variables**; `getpass` shows a hidden input box so secrets aren't printed or saved in the notebook.
- `get_key(name, prompt)` is a reusable helper:
  1. `os.getenv(name)` — check if the env var already exists.
  2. If not, prompt the user with `getpass` to type/paste it.
  3. Store it back into `os.environ` (many SDKs auto-read env vars) and also return it as a Python variable.
- The five calls load keys for **Gemini**, **Aviationstack** (flights), **NewsData.io** (news), **Tavily** (search), and **OpenWeather** (weather).
- **Why this matters:** avoids hard-coding secrets into the notebook — a security best practice.

In [ ]:
import os
from getpass import getpass

def get_key(name, prompt):
    value = os.getenv(name)
    if not value:
        value = getpass(prompt)
    os.environ[name] = value
    return value

GOOGLE_API_KEY = get_key("GOOGL
E_API_KEY", "Gemini API key: ")
AVIATIONSTACK_KEY = get_key("AVIATIONSTACK_KEY", "Aviationstack access key: ")
NEWSDATA_API_KEY = get_key("NEWSDATA_API_KEY", "NewsData.io API key: ")
TAVILY_API_KEY = get_key("TAVILY_API_KEY", "Tavily API key: ")
OPENWEATHER_API_KEY = get_key("OPENWEATHER_API_KEY", "OpenWeather API key: ")

print("API keys loaded.")

### Cell: Initialize the LLM and search client

- Imports: `json`/`requests` for parsing/HTTP; `typing` helpers for the state schema; `StateGraph, START, END` (LangGraph's core building blocks); `ChatGoogleGenerativeAI` (LangChain's Gemini wrapper); `HumanMessage` (a chat message "from the user"); `TavilyClient` (Tavily's client).
- `llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.2, google_api_key=GOOGLE_API_KEY)` creates one reusable LLM object.
  - `temperature=0.2` keeps output fairly deterministic/consistent — important since several prompts ask for structured JSON.
- `tavily = TavilyClient(api_key=TAVILY_API_KEY)` creates one reusable search client.
- Both objects are created **once** here and reused by every agent later.

In [ ]:
import json
import requests
from typing import TypedDict, List, Dict, Any

from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from tavily import TavilyClient

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.2,
    google_api_key=GOOGLE_API_KEY
)

tavily = TavilyClient(api_key=TAVILY_API_KEY)

print("Gemini + Tavily initialized.")

#shared langgraph state

### Cell: Shared LangGraph State — `TravelState`

- `TravelState` is a **TypedDict**: behaves like a normal dict at runtime, but documents the schema (field names + types) for readability/tooling.
- `total=False` means none of the fields are required — the state can start with just a few keys and grow as agents add more.
- Fields fall into three groups:
  1. **Inputs** parsed from the user's request: `origin`, `destination`, IATA codes, dates, `days`, `travelers`, `preferences`.
  2. **Research results** from each specialist agent: `flights`, `hotels`, `news`, `weather`, `cost_research`, `cost` (all flexible `Dict[str, Any]` since each API returns differently-shaped JSON).
  3. **Outputs**: `itinerary`, `final_answer`.
- This single object is passed between every node in the graph — each agent reads what it needs and returns a dict of new/updated fields, which LangGraph merges back in.

In [ ]:
class TravelState(TypedDict, total=False):
    user_request: str

    origin: str
    destination: str
    origin_iata: str
    destination_iata: str
    start_date: str
    end_date: str
    days: int
    travelers: int
    preferences: str

    flights: Dict[str, Any]
    hotels: Dict[str, Any]
    news: Dict[str, Any]
    weather: Dict[str, Any]
    cost_research: Dict[str, Any]
    cost: Dict[str, Any]

    itinerary: str
    final_answer: str

#Supervisor agent
Extract origin , destination , dates, no. of travellers

### Cell: Supervisor Agent — extract structured trip details from free text

- `supervisor_agent(state)` is a LangGraph node: a plain Python function taking the current state.
- The `prompt` is a multi-line f-string instructing the LLM to act as a structured-data extractor:
  - Explicitly says **"Return ONLY valid JSON"** and forbids markdown fences — a common technique for reliably-parseable LLM output.
  - Doubled `{{ }}` are needed because this is an f-string — a literal `{` or `}` must be escaped this way; the single real placeholder is `{state["user_request"]}`.
  - `"Do not invent IATA codes."` is a guardrail against hallucinated airport codes.
- `response = llm.invoke([HumanMessage(content=prompt)])` sends the prompt to Gemini.
- `content = response.content` — then a defensive `isinstance(content, list)` check handles the fact that some SDK responses return a list of content blocks (string or `{"text": ...}` dicts) instead of a plain string; all pieces are joined into one string.
- `raw.replace("```json", "").replace("```", "")` strips markdown code fences in case the model added them anyway, despite instructions.
- `print(...)` lines are for debugging visibility.
- `data = json.loads(raw)` parses the JSON string into a Python dict; `return data` merges these fields (`origin`, `destination`, IATA codes, dates, `days`, `travelers`, `preferences`) into the shared state.

In [ ]:
def supervisor_agent(state: TravelState):

    prompt = f"""
You are the supervisor of a real-time travel multi-agent system.

Extract structured travel requirements from the user's request.

Return ONLY valid JSON.
Do not use markdown.
Do not use ```json.

Format:

{{
  "origin": "city or airport",
  "destination": "city or airport",
  "origin_iata": "IATA code if confidently known, otherwise empty string",
  "destination_iata": "IATA code if confidently known, otherwise empty string",
  "start_date": "YYYY-MM-DD or empty string",
  "end_date": "YYYY-MM-DD or empty string",
  "days": 5,
  "travelers": 2,
  "preferences": "travel preferences"
}}

Do not invent IATA codes.

User request:
{state["user_request"]}
"""

    response = llm.invoke([
        HumanMessage(content=prompt)
    ])

    # Gemini may return content as a string OR a list of content blocks
    content = response.content

    if isinstance(content, list):
        text_parts = []

        for item in content:
            if isinstance(item, str):
                text_parts.append(item)

            elif isinstance(item, dict):
                if "text" in item:
                    text_parts.append(item["text"])

        content = "".join(text_parts)

    raw = content.strip()

    # Remove markdown fences if Gemini adds them
    raw = raw.replace("```json", "").replace("```", "").strip()

    print("SUPERVISOR RESPONSE:")
    print(raw)

    data = json.loads(raw)

    return data

#Test

### Cell: Test the Supervisor Agent in isolation

- Builds a minimal `test_state` containing only `user_request` (a natural-language trip description).
- Calls `supervisor_agent(test_state)` directly — **outside the graph** — a good pattern for unit-testing one agent before wiring the full pipeline.
- Prints the parsed result to confirm the LLM correctly extracted origin, destination, IATA codes, dates, `days`, `travelers`, and `preferences`.

In [ ]:
test_state = {
    "user_request": "I want to travel from Delhi to Goa from 10 September 2026 to 15 September 2026 for 2 people. I prefer budget hotels."
}

result = supervisor_agent(test_state)

print("\nFINAL RESULT:")
print(result)

#Travel agent - Aviationstack

### Cell: Flight Agent — Aviationstack

- `aviationstack_get(params)` is a reusable helper for calling Aviationstack's `/flights` REST endpoint:
  - `params = dict(params)` copies the input dict so the caller's original isn't mutated when we add the API key.
  - `params["access_key"] = AVIATIONSTACK_KEY` — required auth parameter.
  - `requests.get(url, params=params, timeout=30)` sends the GET request; `requests` turns `params` into a query string automatically. `timeout=30` avoids hanging forever.
  - `response.raise_for_status()` raises immediately on HTTP-level errors (4xx/5xx).
  - `if data.get("error"): raise RuntimeError(...)` catches API-level errors embedded in an otherwise-200 response.
- `flight_agent(state)` is the actual graph node:
  - Builds `params` conditionally with `state.get(...)` (returns `None`/falsy if missing, so filters are only added when available) — mapping `origin_iata`→`dep_iata`, `destination_iata`→`arr_iata`, `start_date`→`flight_date`.
  - Calls the helper, then returns a `"flights"` dict recording `source`, the exact `query` sent, the `results` list, and `pagination` info — all merged into shared state.

In [ ]:
def aviationstack_get(params):
    url = "https://api.aviationstack.com/v1/flights"
    params = dict(params)
    params["access_key"] = AVIATIONSTACK_KEY

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if data.get("error"):
        raise RuntimeError(data["error"])

    return data


def flight_agent(state: TravelState):
    params = {"limit": 20}

    if state.get("origin_iata"):
        params["dep_iata"] = state["origin_iata"]

    if state.get("destination_iata"):
        params["arr_iata"] = state["destination_iata"]

    if state.get("start_date"):
        params["flight_date"] = state["start_date"]

    data = aviationstack_get(params)

    return {
        "flights": {
            "source": "Aviationstack",
            "query": params,
            "results": data.get("data", []),
            "pagination": data.get("pagination", {})
        }
    }

#Test the aviationstack directly

### Cell: Test Aviationstack directly

- Calls the low-level `aviationstack_get` helper (not the full agent) with just `limit=5`, to confirm the API key/connection work before relying on it inside the graph.
- Prints how many flights came back and the first flight record as a sanity check of the data shape.

In [ ]:
# Test Aviationstack directly

test_params = {
    "limit": 5
}

data = aviationstack_get(test_params)

print("✅ Aviationstack API is working")
print("Number of flights returned:", len(data.get("data", [])))

print("\nFirst flight:")
if data.get("data"):
    print(data["data"][0])
else:
    print("No flight data returned")

#HOTEL AGENT

### Cell: Hotel Agent — Tavily web search

- `destination = state["destination"]` — uses `[...]` (not `.get()`) deliberately: this agent *requires* a destination, so a missing one raises a clear error instead of silently searching for "None."
- Builds a multi-line search `query` (then `.strip()` trims stray whitespace) asking for hotel prices/reviews/availability for the trip length.
- `tavily.search(query, search_depth="advanced", max_results=8, include_answer=True)`:
  - `search_depth="advanced"` = more thorough (slower/costlier) search.
  - `include_answer=True` asks Tavily to also generate a short synthesized summary.
- Returns a `"hotels"` dict with `source`, `query`, Tavily's `answer` summary, and the raw `results` list.

In [ ]:
def hotel_agent(state: TravelState):
    destination = state["destination"]

    query = f'''
best hotels in {destination}
hotel prices reviews availability
for a {state.get("days", 1)} day trip
'''.strip()

    response = tavily.search(
        query,
        search_depth="advanced",
        max_results=8,
        include_answer=True
    )

    return {
        "hotels": {
            "source": "Tavily",
            "query": query,
            "answer": response.get("answer"),
            "results": response.get("results", [])
        }
    }

### Cell: Test the Hotel Agent

- `pprint` ("pretty print") formats nested data structures more readably than plain `print`.
- Builds a minimal `test_state` (`destination`, `days`), calls `hotel_agent` directly, and prints the source, exact query, Tavily's summarized answer, result count, and full raw results.

In [ ]:
import pprint

# Create a small test State for the Hotel Agent
test_state = {
    "destination": "Manali",
    "days": 5
}

# Execute the Hotel Agent
hotel_result = hotel_agent(test_state)

print("========== HOTEL AGENT TEST ==========")

print("\nSource:")
print(hotel_result["hotels"]["source"])

print("\nQuery sent to Tavily:")
print(hotel_result["hotels"]["query"])

print("\nTavily Answer:")
print(hotel_result["hotels"]["answer"])

print("\nNumber of search results:")
print(len(hotel_result["hotels"]["results"]))

print("\nSearch Results:")
pprint.pp(hotel_result["hotels"]["results"])

#News agent

### Cell: News Agent — NewsData.io

- Calls NewsData.io's `/latest` endpoint directly with `requests` (no separate helper this time).
- `params`: `apikey` (auth), `q` (destination as the search keyword), `language: "en"`, `removeduplicate: 1` (filter near-duplicate articles), `size: 10` (up to 10 articles).
- Same two-layer error handling as the flight agent: `response.raise_for_status()` for HTTP errors, `data.get("status") == "error"` for API-level errors in a 200 response.
- Returns `"news"` with `source`, `query`, the `results` list, and `nextPage` (a pagination token, captured but not used further here).

In [ ]:
def news_agent(state: TravelState):
    destination = state["destination"]

    params = {
        "apikey": NEWSDATA_API_KEY,
        "q": destination,
        "language": "en",
        "removeduplicate": 1,
        "size": 10
    }

    response = requests.get(
        "https://newsdata.io/api/1/latest",
        params=params,
        timeout=30
    )
    response.raise_for_status()
    data = response.json()

    if data.get("status") == "error":
        raise RuntimeError(data)

    return {
        "news": {
            "source": "NewsData.io",
            "query": destination,
            "results": data.get("results", []),
            "nextPage": data.get("nextPage")
        }
    }

### Cell: Test the News Agent

- Same isolated-testing pattern: minimal `test_state`, direct call to `news_agent`, pretty-printed source/query/result-count/results.

In [ ]:
import pprint

# Create a small test State for the News Agent
test_state = {
    "destination": "india"
}

# Execute the News Agent
news_result = news_agent(test_state)

print("========== NEWS AGENT TEST ==========")

print("\nSource:")
print(news_result["news"]["source"])

print("\nQuery sent to NewsData.io:")
print(news_result["news"]["query"])

print("\nNumber of news articles:")
print(len(news_result["news"]["results"]))

print("\nNews Results:")
pprint.pp(news_result["news"]["results"])

#Weather agent

### Cell: Weather Agent — OpenWeather

- Needs **two** API calls because the current-weather endpoint needs coordinates, not a place name:
  1. **Geocoding call** — `/geo/1.0/direct` with `q=destination`, `limit=1` (best match only). `if not locations: raise RuntimeError(...)` handles the case where the name can't be matched to any real place. `location = locations[0]` gives the best match; `lat`/`lon` are extracted from it.
  2. **Current-weather call** — `/data/2.5/weather` using those coordinates, with `units="metric"` for Celsius instead of the default Kelvin.
- Returns `"weather"` with `source`, the resolved `location` (confirms which real place was matched, e.g. which state/country), and `current` (temperature, conditions, humidity, wind, etc.).

In [ ]:
def weather_agent(state: TravelState):
    destination = state["destination"]

    # OpenWeather Geocoding API
    geo_response = requests.get(
        "https://api.openweathermap.org/geo/1.0/direct",
        params={
            "q": destination,
            "limit": 1,
            "appid": OPENWEATHER_API_KEY
        },
        timeout=30
    )
    geo_response.raise_for_status()
    locations = geo_response.json()

    if not locations:
        raise RuntimeError(f"Could not geocode destination: {destination}")

    location = locations[0]
    lat = location["lat"]
    lon = location["lon"]

    # Current weather API
    weather_response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "lat": lat,
            "lon": lon,
            "appid": OPENWEATHER_API_KEY,
            "units": "metric"
        },
        timeout=30
    )
    weather_response.raise_for_status()

    weather = weather_response.json()

    return {
        "weather": {
            "source": "OpenWeather",
            "location": location,
            "current": weather
        }
    }

### Cell: Test the Weather Agent

- Minimal `test_state` with just `destination`, direct call to `weather_agent`, pretty-printed resolved location and current conditions.

In [ ]:
import pprint

# Create a small test State for the Weather Agent
test_state = {
    "destination": "Manali"
}

# Execute the Weather Agent
weather_result = weather_agent(test_state)

print("========== WEATHER AGENT TEST ==========")

print("\nSource:")
print(weather_result["weather"]["source"])

print("\nLocation found:")
pprint.pp(weather_result["weather"]["location"])

print("\nCurrent Weather:")
pprint.pp(weather_result["weather"]["current"])

#cost agent

### Cell: Cost Agent — combining live research with LLM reasoning

- This agent is **hybrid**: it runs its own fresh Tavily search *and* combines it with data already gathered by other agents, then asks the LLM to reason over everything.
- Builds a broad cost-focused `query` (hotels, local transport, attractions, food, activities) scaled to `days`/`travelers`, then runs `tavily.search(...)` the same way the hotel agent does.
- The `synthesis_prompt` is a **grounding / anti-hallucination pattern** worth studying closely:
  - Explicitly instructs: *"Estimate the trip cost using ONLY information available in the retrieved research"* and *"Never fabricate a price."*
  - **Injects real data directly into the prompt** — hotel research from `state`, the fresh `research` just performed, and flight data from `state` — this is the essence of RAG-style prompting: ground the model in real, current facts instead of its (possibly outdated) training data.
  - Requests a strict JSON schema, same technique as the supervisor agent, so the result can be reliably parsed downstream.
  - `flight_cost` explicitly says "unknown unless a real fare was actually found" — again fighting hallucination.
- `response = llm.invoke(...)` sends this large prompt to Gemini. Note: unlike the supervisor agent, this response is **not** parsed with `json.loads()` — it's stored as raw text/content, which is worth discussing (what would need to change to make it reliably machine-readable?).
- Returns **two** state keys at once: `"cost_research"` (raw Tavily research) and `"cost"` (a summary dict with `source`, `research`, and `estimate` = `response.content`).

In [ ]:
def cost_agent(state: TravelState):
    destination = state["destination"]
    days = state.get("days", 1)
    travelers = state.get("travelers", 1)

    query = f'''
current travel cost for {destination}
hotel price per night
local taxi transport cost
tourist attraction ticket prices
food cost per person
activities and sightseeing costs
for {days} days and {travelers} travelers
'''.strip()

    research = tavily.search(
        query,
        search_depth="advanced",
        max_results=10,
        include_answer=True
    )

    synthesis_prompt = f'''
You are the Cost Agent in a travel multi-agent system.

Estimate the trip cost using ONLY information available in the retrieved research.

Destination: {destination}
Days: {days}
Travelers: {travelers}

Hotel research:
{state.get("hotels", {})}

Additional live cost research:
{research}

Flight/aviation data:
{state.get("flights", {})}

Return JSON:
{{
  "hotel_cost": "...",
  "transport_cost": "...",
  "food_cost": "...",
  "activities_cost": "...",
  "flight_cost": "unknown unless a real fare was actually found",
  "total_estimate": "...",
  "currency": "INR",
  "assumptions": ["..."],
  "source_notes": ["..."]
}}

Never fabricate a price.
If a price is unavailable, say "not available from retrieved sources".
'''

    response = llm.invoke([HumanMessage(content=synthesis_prompt)])

    return {
        "cost_research": research,
        "cost": {
            "source": "Tavily + Gemini synthesis",
            "research": research,
            "estimate": response.content
        }
    }

### Cell: Test the Cost Agent

- Builds a `test_state` with fabricated sample `hotels` and `flights` data (simulating what earlier agents would have already produced), then calls `cost_agent` directly and prints both `cost_research` and `cost`.
- In this sample run, note that `estimate` comes back as a list of content blocks (a dict with `type: "text"`, the text itself, and an internal `extras` signature) rather than a plain string — the same "content can be a string or a list" behavior seen with the supervisor agent, which is why `final_agent` later needs the same normalization logic.

In [ ]:
import pprint

# Create a test State for the Cost Agent
test_state = {
    "destination": "Manali",
    "days": 5,
    "travelers": 2,

    "hotels": {
        "source": "Tavily",
        "results": [
            {
                "title": "Sample hotel research",
                "content": "Hotel prices vary depending on season and hotel category."
            }
        ]
    },

    "flights": {
        "source": "Aviationstack",
        "results": []
    }
}

# Execute the Cost Agent
cost_result = cost_agent(test_state)

print("========== COST AGENT TEST ==========")

print("\nCost Research:")
pprint.pp(cost_result.get("cost_research"))

print("\nCost Result:")
pprint.pp(cost_result.get("cost"))

### Cell: Itinerary Agent — turning research into a day-by-day plan

- This is the **synthesis point** of the whole pipeline: by the time this runs, `state` should already contain flights, hotels, weather, news, and cost data (guaranteed by the graph's edges — see the graph-building cell below), so the prompt pulls in **all of it at once**.
- The numbered "Rules" list is more explicit prompt engineering to reduce hallucination and keep output useful:
  1–3. Don't invent facts, hotel availability, or bookable flights.
  4–5. Actively use weather and news data meaningfully.
  6. Stay geographically realistic.
  7. Clearly mark estimates.
- The requested output sections (travel approach, hotel shortlist, itinerary, weather considerations, alerts, budget, assumptions) define the shape of the response.
- `return {"itinerary": response.content}` stores the raw text directly (this one is meant for humans to read, not parsed as JSON).

In [ ]:
#Initnary agent
def itinerary_agent(state: TravelState):
    prompt = f'''
You are the Itinerary Agent.

Create a practical {state.get("days", 1)}-day itinerary.

Trip:
{state["origin"]} -> {state["destination"]}

Travelers: {state.get("travelers", 1)}
Dates: {state.get("start_date", "")} to {state.get("end_date", "")}
Preferences: {state.get("preferences", "")}

LIVE FLIGHT DATA:
{state.get("flights", {})}

LIVE HOTEL RESEARCH:
{state.get("hotels", {})}

CURRENT WEATHER:
{state.get("weather", {})}

CURRENT NEWS:
{state.get("news", {})}

LIVE COST RESEARCH:
{state.get("cost", {})}

Rules:
1. Do not invent live facts.
2. Do not claim a hotel is available unless the source says so.
3. Do not claim a flight is bookable.
4. Make the plan weather-aware.
5. Use current news to flag relevant travel disruptions or events.
6. Keep the itinerary realistic geographically.
7. Clearly mark anything that is an estimate.

Return:
- recommended travel approach
- hotel shortlist
- day-by-day itinerary
- weather considerations
- current travel/news alerts
- estimated budget
- assumptions
'''

    response = llm.invoke([HumanMessage(content=prompt)])
    return {"itinerary": response.content}

#final agent

### Cell: Final Agent — assembling the user-facing answer

- The emoji `print` statements act as simple progress logs — useful for a live/teaching demo to see the pipeline moving through this final, often slower, step.
- The prompt is similar to the itinerary agent's, but its job is different: rather than *generating* a new itinerary, it **assembles/formats** everything already gathered (including the itinerary text already produced) into one final, polished, emoji-labeled answer.
- Same anti-hallucination guardrails repeated ("Do not invent information/prices").
- `content = response.content` then the **same normalization pattern** as the supervisor agent: if `content` is a list, loop through it collecting strings and `{"text": ...}` dict values into one combined string. This is the second (and last) place the raw model response is read directly for display to a human, so it needs the same defensive handling.
- `return {"final_answer": content}` stores the finished, human-readable travel plan — exactly the key the last cell of the notebook reads (`result["final_answer"]`).

In [ ]:
def final_agent(state: TravelState):

    print("🎯 FINAL AGENT STARTED")

    prompt = f"""
You are the final travel assistant.

Create the final travel plan using the information
collected by the other agents.

USER REQUEST:
{state.get("user_request", "")}

FLIGHTS:
{state.get("flights", {})}

HOTELS:
{state.get("hotels", {})}

NEWS:
{state.get("news", {})}

WEATHER:
{state.get("weather", {})}

COST:
{state.get("cost", {})}

ITINERARY:
{state.get("itinerary", "")}

Create a final travel plan containing:

✈️ Flight information
🏨 Hotel recommendations
🌤️ Weather
📰 Important news
💰 Estimated cost
🗺️ Day-by-day itinerary

Important:
- Do not invent information.
- Do not invent prices.
- Clearly mention estimates.
- Use the information provided by the agents.
"""

    print("🤖 Calling Gemini...")

    response = llm.invoke([
        HumanMessage(content=prompt)
    ])

    print("🤖 Gemini response received")

    content = response.content

    # Gemini can return string OR list
    if isinstance(content, list):

        text = ""

        for item in content:

            if isinstance(item, str):
                text += item

            elif isinstance(item, dict):
                if "text" in item:
                    text += item["text"]

        content = text

    print("🎯 FINAL AGENT COMPLETED")

    return {
        "final_answer": content
    }

### Cell: Building the Graph

- `builder = StateGraph(TravelState)` creates a new graph builder tied to the `TravelState` schema.
- `builder.add_node(name, function)` registers each Python function as a named node (the name string is what's used to wire edges — it doesn't have to match the function name).
- `builder.add_edge(from_node, to_node)` defines a directed connection: "once `from_node` finishes, `to_node` becomes eligible to run." `START` and `END` are special sentinel nodes marking entry/exit.
- Walking through the wiring — this is a **fan-out / fan-in** shape:
  1. `START → supervisor`: every run begins here.
  2. **Fan-out:** `supervisor` connects to four independent agents (`flight_agent`, `hotel_agent`, `news_agent`, `weather_agent`) — since none depend on each other, LangGraph can run them **in parallel**.
  3. **Partial fan-in:** both `flight_agent` and `hotel_agent` feed `cost_agent`, which waits for both before running.
  4. **Full fan-in:** all five prior agents feed `itinerary_agent`, which waits for everything before it runs, ensuring it has the complete picture.
  5. `itinerary_agent → final_agent → END`: a simple linear finish.
- `graph = builder.compile()` finalizes the graph into an executable object — the comment `# IMPORTANT: compile again` is a reminder that any time nodes/edges change, `.compile()` must be re-run for the changes to take effect.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TravelState)

builder.add_node("supervisor", supervisor_agent)
builder.add_node("flight_agent", flight_agent)
builder.add_node("hotel_agent", hotel_agent)
builder.add_node("news_agent", news_agent)
builder.add_node("weather_agent", weather_agent)
builder.add_node("cost_agent", cost_agent)
builder.add_node("itinerary_agent", itinerary_agent)
builder.add_node("final_agent", final_agent)

# Start
builder.add_edge(START, "supervisor")

# Supervisor → specialist agents
builder.add_edge("supervisor", "flight_agent")
builder.add_edge("supervisor", "hotel_agent")
builder.add_edge("supervisor", "news_agent")
builder.add_edge("supervisor", "weather_agent")

# Flight + Hotel → Cost
builder.add_edge("flight_agent", "cost_agent")
builder.add_edge("hotel_agent", "cost_agent")

# All research → Itinerary
builder.add_edge("flight_agent", "itinerary_agent")
builder.add_edge("hotel_agent", "itinerary_agent")
builder.add_edge("news_agent", "itinerary_agent")
builder.add_edge("weather_agent", "itinerary_agent")
builder.add_edge("cost_agent", "itinerary_agent")

# Itinerary → Final
builder.add_edge("itinerary_agent", "final_agent")

# Final → End
builder.add_edge("final_agent", END)

# IMPORTANT: compile again
graph = builder.compile()

print("✅ Graph rebuilt successfully")

### Cell: Visualizing the Graph

- `graph.get_graph()` extracts a graph-structure representation from the compiled object.
- `.draw_mermaid_png()` renders it as a PNG using Mermaid (diagram-as-code), displayed inline via `IPython.display.Image`/`display()`.
- The `try/except` is a fallback: PNG rendering needs an internet call to a rendering service, which might fail in restricted environments. If it fails, the code prints the raw Mermaid **text** definition instead (`draw_mermaid()`), so the graph structure is still visible as text.
- Seeing the actual flowchart makes the fan-out/fan-in structure from the previous cell much easier to understand visually.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

### Cell: Running the Full Pipeline

- `user_request` is a natural-language description of what the user wants — the kind of free text a real end user might type.
- `graph.invoke({"user_request": user_request})` runs the **entire compiled graph** from start to finish in one call:
  - Starts with an initial state containing only `user_request`.
  - LangGraph automatically executes nodes in the correct order based on the edges defined earlier, merging each node's returned dict into the running state, and running independent nodes concurrently where possible.
  - `result` is the **final state** after every node (through `final_agent`) has run — containing every field accumulated along the way.
- `print(result["final_answer"])` — pulls out just the polished, human-readable travel plan and prints it: this is what an end user of the finished application would actually see.
- The interleaved debug print statements you'll see when running this (`SUPERVISOR RESPONSE:`, `🎯 FINAL AGENT STARTED`, etc.) are a visible trace of the graph executing node by node — a good way to observe the graph's real execution order in practice, not just in theory.

In [ ]:
user_request = '''
Plan a 5-day trip from Delhi to Manali for 2 travelers.
I want comfortable hotels, scenic places, local food and a moderate budget.
Use current flight information, current hotel information, current weather,
recent news and current travel costs. Then create a day-by-day itinerary.
'''

result = graph.invoke({
    "user_request": user_request
})

print(result["final_answer"])